In [1]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/humanoid')

import numpy as np
import mujoco
import mujoco.viewer
from scipy.spatial import ConvexHull

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer
from control import (
    WholeBodyController,
    SimplifiedModelControl, TrajectoryOptimization,
    MotionPlanner,
)

%load_ext autoreload
%autoreload 2

In [2]:
URDF = '/home/frlab/mj_opt/xmls/systems/g1_description/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/xmls/systems/g1_description'
MJCF = '/home/frlab/mj_opt/xmls/systems/g1/scene_29dof.xml'

In [3]:
# ── Pinocchio wrapper 초기화 ──
wrapper = Pinocchio_Wrapper(URDF, PKG)

# ── MuJoCo kernel 초기화 (joint 순서는 pinocchio 기준) ──
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"R_foot": "right_ankle_roll_link", "L_foot": "left_ankle_roll_link",})

# ── WBC 초기화 (게인은 여기서 빠르게 튜닝) ──
wbc = WholeBodyController(
    wrapper,
    kp_com=100.0,    # 워킹은 30~50 권장 (현재는 standing 게인)
    kd_com=20.0,
    mu=0.5,
    nc=12,
)

print(f"nv={wrapper.nv}, na={wrapper.na}, mass={wrapper.mass:.2f} kg")

nv=35, na=29, mass=35.12 kg


In [4]:
# ── 워킹 파라미터 ──
DT          = 0.001
N_STEPS     = 20
STEP_TIME   = 1.0
DSP_TIME    = 0.1
INIT_DSP_EXTRA = 0.12
STEP_LENGTH = 0.15
STEP_WIDTH  = 0.1185
STEP_HEIGHT = 0.05

# CoM 관련
Z_C_OFFSET  = -0.1     # 초기 CoM 높이 기준 오프셋 (낮춤 = 무릎 굽힘)

# DCM 컨트롤 게인
K_DCM       = 2.0
KI_DCM      = 0.0
K_ZMP       = 0.0      # 0: ZMP 측정 비활성 (Stage 1+2)
K_COM       = 1.0

In [5]:
# ── 초기 상태 측정 ──
kernel.push_to_wrapper(wrapper)
com_init = wrapper.pos_com_world.copy()
lf_init  = wrapper.oM_Lfoot.translation.copy()
rf_init  = wrapper.oM_Rfoot.translation.copy()

z_c = com_init[2] + Z_C_OFFSET
print(f"초기 CoM={com_init},  walk z_c={z_c:.3f}")
print(f"L_foot={lf_init},  R_foot={rf_init}")

# ── (1) CoM/DCM/footstep 사전계산 ──
traj_opt = TrajectoryOptimization(
    z_c=z_c,
    step_time=STEP_TIME,
    dsp_time=DSP_TIME,
    dt=DT,
    init_dsp_extra=INIT_DSP_EXTRA,
)
footsteps, ref_dcm, ref_dcm_vel, ref_com, ref_com_vel = \
    traj_opt.compute_all_trajectories(
        n_steps=N_STEPS,
        step_length=STEP_LENGTH,
        step_width=STEP_WIDTH,
        init_com=np.array([com_init[0], com_init[1], z_c]),
    )
ref_com[:, 2] = z_c

# ── (2) 발 궤적 (위치/속도/가속도 모두 사전계산) ──
mp = MotionPlanner(default_step_height=STEP_HEIGHT)
lf_traj, rf_traj, lf_vel, rf_vel, lf_acc, rf_acc = mp.build_foot_trajectories(
    footsteps, lf_init, rf_init,
    step_time=STEP_TIME,
    dsp_time=DSP_TIME,
    init_dsp_extra=INIT_DSP_EXTRA,
    dt=DT,
    step_height=STEP_HEIGHT,
)

T_TOTAL = len(ref_com) * DT
print(f"trajectory: {len(ref_com)} samples, total {T_TOTAL:.2f}s, footsteps={len(footsteps)}")

# ── DCM controller 초기화 ──
lipm_ctrl = SimplifiedModelControl(
    z_c=z_c,
    k_dcm=K_DCM, ki_dcm=KI_DCM, k_zmp=K_ZMP, k_com=K_COM,
    dt=DT,
)

초기 CoM=[1.95688682e-02 7.21710511e-05 7.21818268e-01],  walk z_c=0.622
L_foot=[-2.32609678e-06  1.18506455e-01  3.61362476e-02],  R_foot=[-2.32609678e-06 -1.18506455e-01  3.61362476e-02]
trajectory: 20120 samples, total 20.12s, footsteps=20


In [6]:
# ── Task-space PD 게인 (워킹용 보조 task) ─────────────────────
# 현재 WBC.compute()는 이 task들을 직접 받지 못함.
# 여기서 *desired acceleration*만 미리 계산해두고, 후속 단계에서 WBC에 주입할 예정.

# 1) Swing foot tracking (Operational space PD)
#    ẍ_sw_des = ẍ_ref + Kp·(x_ref - x) + Kd·(ẋ_ref - ẋ)
KP_SWING = 200.0
KD_SWING =  20.0

# 2) Base orientation (Pelvis upright 유지)
#    ω̇_des = -Kp·log(R·R_des^T) - Kd·ω
KP_BASE_R = 100.0
KD_BASE_R =  10.0
R_BASE_DES = np.eye(3)   # pelvis 정위 (초기 자세 유지)

# 3) Posture (Joint q_neutral regulation, weight 작게)
#    q̈_des = Kp·(q_ref - q) - Kd·q̇
KP_POST = 20.0
KD_POST =  4.0
# q_ref는 init 시점의 actuated joint 값을 그대로 유지
q_post_ref = wrapper.q[7:].copy()   # nq=36에서 free-flyer(7) 제외 → na=29

print(f"posture ref: {q_post_ref.shape}, base_R_des=I")

posture ref: (29,), base_R_des=I


In [7]:
import pinocchio as pin

# ── Task-space PD 헬퍼들 ────────────────────────────────────────
# 모두 wrapper.update_model() 직후 호출 가정.
# 반환은 *desired acceleration* (3,) 또는 (na,).
# ẍ_des = ẍ_ref + Kp·(x_ref - x) + Kd·(ẋ_ref - ẋ)

def swing_foot_pd(key, x_ref, v_ref, a_ref, kp=KP_SWING, kd=KD_SWING):
    """
    key   : 'L_foot' or 'R_foot'  (wrapper.fid 키)
    x_ref : (3,) world position ref
    v_ref : (3,) world linear velocity ref
    a_ref : (3,) world linear acceleration ref (feedforward)
    """
    fid = wrapper.fid[key]
    p = wrapper.data.oMf[fid].translation
    v = pin.getFrameVelocity(wrapper.model, wrapper.data, fid,
                             pin.ReferenceFrame.LOCAL_WORLD_ALIGNED).linear
    return a_ref + kp * (x_ref - p) + kd * (v_ref - v)


def base_orientation_pd(R_des=R_BASE_DES, kp=KP_BASE_R, kd=KD_BASE_R):
    """
    Pelvis 정위 유지. Returns (3,) angular acceleration des (world frame).
    """
    R_cur = wrapper.oMb.rotation
    # log(R_cur · R_des^T) → world-frame rotation error vector
    R_err = R_cur @ R_des.T
    e_R = pin.log3(R_err)
    # base angular velocity (world frame — wrapper의 _dq[3:6]은 body frame 각속도)
    omega_b_body = wrapper._dq[3:6]
    omega_w = R_cur @ omega_b_body
    return -kp * e_R - kd * omega_w


def posture_pd(q_ref=q_post_ref, kp=KP_POST, kd=KD_POST):
    """
    Joint posture regulation. Returns (na,) joint acceleration des.
    """
    q_act  = wrapper.q[7:]      # actuated joint pos (na,)
    dq_act = wrapper.dq[6:]     # actuated joint vel (na,) — floating base 6 제외
    return kp * (q_ref - q_act) - kd * dq_act

In [8]:
N_TRAJ = len(ref_com)

def _idx(t):
    """sim 시간 → trajectory 인덱스 (마지막 샘플에서 hold)"""
    i = int(t / DT)
    return min(i, N_TRAJ - 1)


# 디버깅용 — 매 control 호출 시 task acc desired 보관
last_task = {'sw_L': None, 'sw_R': None, 'base_ang': None, 'post': None}


def on_control(t):
    i = _idx(t)

    # 1) MuJoCo → Pinocchio 동기화
    kernel.push_to_wrapper(wrapper)

    # 2) DCM controller: ref_com → desired CoM velocity
    desired_vel_xy, _, _ = lipm_ctrl.control_step(
        meas_com_pos = wrapper.pos_com_world,
        meas_com_vel = wrapper.vel_com_world,
        ref_dcm      = ref_dcm[i],
        ref_dcm_vel  = ref_dcm_vel[i],
        ref_com_pos  = ref_com[i],
        ref_com_vel  = ref_com_vel[i],
    )

    # 3) Task-space PD 계산 (현재는 WBC에 주입 안 함, 디버깅 목적)
    last_task['sw_L']     = swing_foot_pd('L_foot', lf_traj[i], lf_vel[i], lf_acc[i])
    last_task['sw_R']     = swing_foot_pd('R_foot', rf_traj[i], rf_vel[i], rf_acc[i])
    last_task['base_ang'] = base_orientation_pd()
    last_task['post']     = posture_pd()

    # 4) WBC 입력: 3D CoM target (xy는 ref, z는 z_c 고정)
    com_des_3d = np.array([ref_com[i, 0], ref_com[i, 1], z_c])
    com_dot_des_3d = np.array([desired_vel_xy[0], desired_vel_xy[1], 0.0])

    # 5) WBC → torque (현재는 CoM PD만 — 위 task들은 아직 미반영)
    tau = wbc.compute(com_des_3d, com_dot_des_3d)
    kernel.apply_torque_pin(tau)


def on_render(t, visualizer):
    i = _idx(t)
    I3 = np.eye(3)

    # 현재 base + 목표 CoM
    visualizer.draw_axes(wrapper.oMb.translation, wrapper.oMb.rotation, size=0.15)
    com_des_3d = np.array([ref_com[i, 0], ref_com[i, 1], z_c])
    visualizer.draw_axes(com_des_3d, I3, size=0.15)

    # 발 reference 좌표계
    visualizer.draw_axes(lf_traj[i], I3, size=0.15)
    visualizer.draw_axes(rf_traj[i], I3, size=0.15)

    # footstep 좌표계
    for fx, fy in footsteps:
        visualizer.draw_axes(np.array([fx, fy, 0.005]), I3, size=0.15)

    # ConvexHull (support polygon)
    _, points = kernel.get_foot_contact_state()
    if len(points) >= 3:
        pts = np.array(points)
        try:
            hull = ConvexHull(pts[:, :2])
            z = pts[:, 2].mean()
            verts = pts[hull.vertices]
            for j in range(len(verts)):
                p1 = np.array([verts[j][0], verts[j][1], z])
                p2 = np.array([verts[(j+1) % len(verts)][0], verts[(j+1) % len(verts)][1], z])
                visualizer.draw_line(p1, p2, rgba=(0, 1, 0, 1), thickness=0.005)
        except Exception:
            pass

In [9]:
# ── 시뮬 실행 ──
# ctrl_hz=500, render_hz=60  (viewer 닫으면 종료)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=on_control, on_render=on_render)